# Una solución empresarial completa

## Ahora llevaremos nuestro proyecto del día 1 al siguiente nivel

### DESAFÍO EMPRESARIAL:

Crear un producto que genere un folleto para una empresa que se utilizará para posibles clientes, inversores y posibles reclutas.

Se nos proporcionará un nombre de empresa y su sitio web principal.

Consulte el final de este cuaderno para ver ejemplos de aplicaciones empresariales del mundo real.

Y recuerde: ¡siempre estoy disponible si tiene problemas o ideas! No dude en comunicarse conmigo.

In [1]:
# imports
# Si esto falla, verifica que esté ejecutándose desde un entorno "activado" con (llms) en el símbolo del sistema

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Inicialización y constantes and constants

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")

MODEL = 'gpt-4o-mini'
openai = OpenAI()

La clave de API parece buena


In [3]:
# La clase para representar una Página Web

class Website:
    """
    Una clase de utilidad para representar un sitio web que hemos scrappeado, ahora con enlaces
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "Sin título"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Título de la Web:\n{self.title}\nContenido de la Web:\n{self.text}\n\n"

In [5]:
frog = Website("https://jhymer.dev")
print(frog.get_contents())
frog.links

Título de la Web:
Desarrollo y Más
Contenido de la Web:
Home
Desarrollo y Más
docker
Optimizar un Entorno de Desarrollo con Docker y Docker-Compose
Docker ha evolucionado a tal punto de ser una herramienta indispensable no solo para el despliegue de aplicaciones sino también, como veremos en este caso, como una herramienta que facilite
Jhymer Martínez
Jhymer Martínez
24 May 2020
•
5 min read
javascript
¿Qué son y para qué sirven los callbacks en JavaScript?
Un callback (llamada de vuelta) en términos comunes es una función que recibe otra función como argumento y la ejecuta en un determinado momento. Para entenderlo mejor usaremos un ejemplo
Jhymer Martínez
Jhymer Martínez
14 Apr 2020
•
3 min read
productivity
6 herramientas para trabajar remotamente de manera efectiva
En muchas ocasiones es necesario trabajar remotamente por lo que surge la necesidad de lograr una comunicación efectiva con el equipo y a la vez gestionar adecuadamente nuestros proyectos, para
Jhymer Martínez
Jhymer Mar

['https://jhymer.dev',
 'https://jhymer.dev/',
 'https://www.facebook.com/ghost',
 'https://twitter.com/JhymerMartinez',
 'https://feedly.com/i/subscription/feed/https://jhymer.dev/rss/',
 '/docker-compose-entorno-desarrollo/',
 '/docker-compose-entorno-desarrollo/',
 '/author/jhymer/',
 '/author/jhymer/',
 '/que-son-y-para-que-sirven-los-callbacks-en-javascript/',
 '/que-son-y-para-que-sirven-los-callbacks-en-javascript/',
 '/author/jhymer/',
 '/author/jhymer/',
 '/herramientas-trabajar-desde-casa/',
 '/herramientas-trabajar-desde-casa/',
 '/author/jhymer/',
 '/author/jhymer/',
 '/integrar-jest-con-enzyme-en-un-proyecto-react-create-react-app/',
 '/integrar-jest-con-enzyme-en-un-proyecto-react-create-react-app/',
 '/author/jhymer/',
 '/author/jhymer/',
 'https://jhymer.dev',
 'https://jhymer.dev/',
 'https://www.facebook.com/ghost',
 'https://twitter.com/JhymerMartinez',
 'https://feedly.com/i/subscription/feed/https://jhymer.dev/rss/',
 'https://jhymer.dev',
 'https://jhymer.dev',
 '

## Primer paso: hacer que GPT-4o-mini determine qué enlaces son relevantes

### Usar una llamada a gpt-4o-mini para leer los enlaces en una página web y responder en JSON estructurado.
Debería decidir qué enlaces son relevantes y reemplazar los enlaces relativos como "/about" con "https://company.com/about".
Usaremos "one shot prompting" en las que proporcionamos un ejemplo de cómo debería responder en la solicitud.

Este es un excelente caso de uso para un LLM, porque requiere una comprensión matizada. Imagínate intentar programar esto sin LLMs analizando la página web: ¡sería muy difícil!

Nota al margen: existe una técnica más avanzada llamada "Salidas estructuradas" en la que requerimos que el modelo responda de acuerdo con una especificación. Cubrimos esta técnica en la Semana 8 durante nuestro proyecto autónomo de inteligencia artificial Agentic.

In [6]:
link_system_prompt = "Se te proporciona una lista de enlaces que se encuentran en una página web. \
Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, \
como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.\n"
link_system_prompt += "Debes responder en JSON como en este ejemplo:"
link_system_prompt += """
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}
"""

In [7]:
print(link_system_prompt)

Se te proporciona una lista de enlaces que se encuentran en una página web. Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.
Debes responder en JSON como en este ejemplo:
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}



In [8]:
def get_links_user_prompt(website):
    user_prompt = f"Aquí hay una lista de enlaces de la página web {website.url} - "
    user_prompt += "Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. \
No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.\n"
    user_prompt += "Links (puede que algunos sean links relativos):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [9]:
print(get_links_user_prompt(frog))

Aquí hay una lista de enlaces de la página web https://jhymer.dev - Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.
Links (puede que algunos sean links relativos):
https://jhymer.dev
https://jhymer.dev/
https://www.facebook.com/ghost
https://twitter.com/JhymerMartinez
https://feedly.com/i/subscription/feed/https://jhymer.dev/rss/
/docker-compose-entorno-desarrollo/
/docker-compose-entorno-desarrollo/
/author/jhymer/
/author/jhymer/
/que-son-y-para-que-sirven-los-callbacks-en-javascript/
/que-son-y-para-que-sirven-los-callbacks-en-javascript/
/author/jhymer/
/author/jhymer/
/herramientas-trabajar-desde-casa/
/herramientas-trabajar-desde-casa/
/author/jhymer/
/author/jhymer/
/integrar-jest-con-enzyme-en-un-proyecto-react-create-react-app/
/integrar-jest-con-enzyme-en-un-proyecto-react-create-react-app/
/author/

In [10]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [11]:
anthropic = Website("https://anthropic.com")
anthropic.links

[]

In [12]:
get_links("https://jhymer.dev")

{'links': [{'type': 'Página Principal', 'url': 'https://jhymer.dev'},
  {'type': 'Página de Facebook', 'url': 'https://www.facebook.com/ghost'},
  {'type': 'Página de Twitter', 'url': 'https://twitter.com/JhymerMartinez'},
  {'type': 'Página de Herramientas para Trabajar',
   'url': 'https://jhymer.dev/herramientas-trabajar-desde-casa/'}]}

In [13]:
#get_links("https://cursos.frogamesformacion.com")
#get_links("https://anthropic.com")


## Segundo paso: ¡crea el folleto!

Reúne todos los detalles en otro mensaje para GPT4-o

In [14]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Links encontrados:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [15]:
#print(get_all_details("https://anthropic.com"))

print(get_all_details("https://jhymer.dev"))

Links encontrados: {'links': [{'type': 'Página Principal', 'url': 'https://jhymer.dev'}, {'type': 'Página de Autor', 'url': 'https://jhymer.dev/author/jhymer/'}, {'type': 'Página de Herramientas', 'url': 'https://jhymer.dev/herramientas-trabajar-desde-casa/'}, {'type': 'Página de Integración Jest', 'url': 'https://jhymer.dev/integrar-jest-con-enzyme-en-un-proyecto-react-create-react-app/'}]}
Landing page:
Título de la Web:
Desarrollo y Más
Contenido de la Web:
Home
Desarrollo y Más
docker
Optimizar un Entorno de Desarrollo con Docker y Docker-Compose
Docker ha evolucionado a tal punto de ser una herramienta indispensable no solo para el despliegue de aplicaciones sino también, como veremos en este caso, como una herramienta que facilite
Jhymer Martínez
Jhymer Martínez
24 May 2020
•
5 min read
javascript
¿Qué son y para qué sirven los callbacks en JavaScript?
Un callback (llamada de vuelta) en términos comunes es una función que recibe otra función como argumento y la ejecuta en un dete

In [16]:
system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa\
y crea un folleto breve sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
Incluye detalles sobre la cultura de la empresa, los clientes, las carreras/empleos y los cursos/packs para futuros empleos si tienes la información."

# O descomenta las líneas a continuación para obtener un folleto más humorístico: esto demuestra lo fácil que es incorporar el "tono":

# system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa \
# y crea un folleto breve, divertido y gracioso sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
#Incluye detalles sobre la cultura de la empresa, los clientes y los cursos/packs para futuros empleos si tienes la información."


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"Estás mirando una empresa llamada: {company_name}\n"
    user_prompt += f"Aquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] # Truncar si tiene más de 20.000 caracteres
    return user_prompt

In [18]:
#get_brochure_user_prompt("Anthropic", "https://anthropic.com")
get_brochure_user_prompt("Jhymer", "https://jhymer.dev")

Links encontrados: {'links': [{'type': 'Pagina Principal', 'url': 'https://jhymer.dev'}, {'type': 'Pagina de Autor', 'url': 'https://jhymer.dev/author/jhymer/'}, {'type': 'Herramientas para Trabajar', 'url': 'https://jhymer.dev/herramientas-trabajar-desde-casa/'}, {'type': 'Integrar Jest con Enzyme', 'url': 'https://jhymer.dev/integrar-jest-con-enzyme-en-un-proyecto-react-create-react-app/'}]}


'Estás mirando una empresa llamada: Jhymer\nAquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\nLanding page:\nTítulo de la Web:\nDesarrollo y Más\nContenido de la Web:\nHome\nDesarrollo y Más\ndocker\nOptimizar un Entorno de Desarrollo con Docker y Docker-Compose\nDocker ha evolucionado a tal punto de ser una herramienta indispensable no solo para el despliegue de aplicaciones sino también, como veremos en este caso, como una herramienta que facilite\nJhymer Martínez\nJhymer Martínez\n24 May 2020\n•\n5 min read\njavascript\n¿Qué son y para qué sirven los callbacks en JavaScript?\nUn callback (llamada de vuelta) en términos comunes es una función que recibe otra función como argumento y la ejecuta en un determinado momento. Para entenderlo mejor usaremos un ejemplo\nJhymer Martínez\nJhymer Martínez\n14 Apr 2020\n•\n3 min read\nproductivity\n6 herramientas para trabajar remotamente

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
#create_brochure("Anthropic", "https://anthropic.com")

create_brochure("Jhymer", "https://jhymer.dev")

Links encontrados: {'links': [{'type': 'Pagina principal', 'url': 'https://jhymer.dev'}, {'type': 'Perfil de autor', 'url': 'https://jhymer.dev/author/jhymer/'}, {'type': 'Herramientas para trabajar desde casa', 'url': 'https://jhymer.dev/herramientas-trabajar-desde-casa/'}, {'type': 'Integrar Jest con Enzyme en React', 'url': 'https://jhymer.dev/integrar-jest-con-enzyme-en-un-proyecto-react-create-react-app/'}]}


# Folleto de Empresa: Jhymer Martínez - Desarrollo y Más

## Sobre Jhymer Martínez

Jhymer es un innovador en el campo del desarrollo de software, especializado en herramientas modernas para optimizar el trabajo en entornos digitales. Con un enfoque particular en tecnologías como Docker, JavaScript y React, la empresa se dedica a proporcionar contenido educativo y soluciones efectivas para desarrolladores y empresas que buscan mejorar su productividad.

---

## Cultura de la Empresa

Jhymer Martínez fomenta una cultura de innovación y aprendizaje continuo. Creemos en el trabajo en equipo y en la colaboración, especialmente en el contexto del trabajo remoto. La empresa promueve el uso de herramientas digitales que faciliten la comunicación y gestión de proyectos, asegurando un entorno de trabajo flexible y adaptado a las necesidades de cada empleado.

---

## Nuestros Clientes

Atendemos a una amplia gama de clientes, que van desde autónomos hasta pequeñas y grandes empresas tecnológicas. Nuestros servicios están diseñados para ayudar a nuestros clientes a optimizar su flujo de trabajo y mejorar la calidad de sus productos mediante el uso de las últimas tecnologías y metodologías ágiles.

---

## Carreras y Oportunidades

En Jhymer, ofrecemos un entorno laboral óptimo para aquellos apasionados del desarrollo y la tecnología. Buscamos talento que comparta nuestro compromiso con la innovación y el aprendizaje continuo. Ofrecemos:

- **Oportunidades de crecimiento profesional**: Ayudamos a nuestros empleados a alcanzar su máximo potencial.
- **Proyectos desafiantes**: Trabaja con tecnologías de vanguardia en un entorno colaborativo.

Si estás interesado en unirte a nuestro equipo, no dudes en ponerte en contacto con nosotros.

---

## Cursos y Packs para Futuros Empleos

Jhymer también ofrece recursos educativos diseñados para preparar a futuros talentos en el campo del desarrollo. Estos incluyen:

- **Cursos sobre Docker y Docker-Compose**: Aprende a optimizar entornos de desarrollo y despliegue de aplicaciones.
- **JavaScript y React**: Cursos que exploran el funcionamiento de callbacks y pruebas unitarias utilizando Jest y Enzyme en proyectos React.
- **Herramientas para la Productividad**: Capacitación sobre herramientas digitales como Slack, Trello, Google Hangouts y más, que facilitan el trabajo remoto.

---

Para más información sobre nuestros servicios y cómo podemos colaborar, visita [Desarrollo y Más](#) o síguenos en nuestras redes sociales.

**Conéctate con nosotros y lleva tu carrera al siguiente nivel!**

In [27]:
create_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

Links encontrados: {'links': [{'type': 'Pagina Sobre nosotros', 'url': 'https://cursos.frogamesformacion.com/pages/rutas'}, {'type': 'Pagina de Instructores', 'url': 'https://cursos.frogamesformacion.com/pages/instructores'}, {'type': 'Pagina de Certificaciones', 'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'}, {'type': 'Pagina de Clientes', 'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'}, {'type': 'Pagina para Empresas', 'url': 'https://cursos.frogamesformacion.com/pages/frogames-para-empresas'}, {'type': 'Pagina de Premios', 'url': 'https://cursos.frogamesformacion.com/pages/premios'}, {'type': 'Pagina de Afiliados', 'url': 'https://cursos.frogamesformacion.com/pages/afiliados'}, {'type': 'Pagina de Cursos', 'url': 'https://cursos.frogamesformacion.com/collections'}]}


# Folleto Informativo de Frogames Formación

## Sobre Frogames
Frogames Formación es una innovadora plataforma de educación en línea premiada como la "Enseñanza online de datos y competencias digitales más innovadora de Europa, 2023". Fundada por **Juan Gabriel Gomila** y **María Santos**, nuestra misión es ofrecer formación de calidad para ayudar a los estudiantes a convertirse en expertos en áreas clave como:
- Programación de Videojuegos
- Inteligencia Artificial
- Data Science
- Desarrollo de Apps
- Machine Learning

Con más de **500,000 estudiantes satisfechos** globalmente, Frogames ha marcado un camino significativo en la educación en línea en habla hispana.

## Cultura de la Empresa
En Frogames, creemos en una cultura educativa inclusiva y divertida. Promovemos un ambiente donde el aprendizaje se lleva de manera amena y eficaz. Nuestros estudiantes destacan el entusiasmo de nuestra comunidad, así como el apoyo continuo que reciben de instructores altamente capacitados.

La innovación y la mejora constante son partícipes de nuestra esencia; actualizamos nuestros cursos regularmente para mantenerlos alineados con las últimas tendencias y tecnologías.

## Ofertas para Clientes
- **Cursos Online**: Acceso a una amplia gama de cursos que abarcan fundamentos y especializaciones en diversas áreas.
- **Certificaciones Blockchain**: Al finalizar nuestros cursos, los estudiantes obtienen certificaciones digitales validadas por tecnología blockchain, mejorando su CV y visibilidad profesional.
- **Rutas de Aprendizaje**: Rutas temáticas diseñadas para facilitar el aprendizaje progresivo en áreas como matemáticas, programación y desarrollo de videojuegos.

### Cursos Destacados
- **Machine Learning de la A a la Z**
- **Curso Completo de Unreal Engine 5**
- **Introducción a C# para Desarrolladores de Videojuegos**
- **Trading Algorítmico para Principiantes**

## Oportunidades de Carrera
Frogames busca personas apasionadas por la educación y tecnología que deseen unirse a nuestro equipo. Ofrecemos diferentes caminos profesionales, desde la docencia hasta el desarrollo de contenido educativo. También brindamos un programa de afiliados, donde puedes ganar comisiones compartiendo nuestros cursos.

## Formación para Empresas
Frogames Formación también ofrece paquetes educativos diseñados para empresas, ayudando a elevar las competencias digitales de los empleados y proporcionando una ventaja competitiva en el mercado.

## Un Futuro Brillante a Tu Alcance
Comienza a aprender hoy con nosotros. Aprovecha nuestras suscripciones y descuentos; actualmente estamos ofreciendo un **60% de descuento** en todos nuestros cursos con el código **BLACKFRIDAY**.

¡Únete a la comunidad Frogames y transforma tu vida profesional a través de la educación!

---

Para más información, visita [Frogames Formación](https://www.frogamesformacion.com) y comienza tu viaje de aprendizaje hoy mismo.

## Por último, una pequeña mejora

Con un pequeño ajuste, podemos cambiar esto para que los resultados se transmitan desde OpenAI,
con la animación de máquina de escribir habitual


In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
#stream_brochure("Anthropic", "https://anthropic.com")
stream_brochure("Jhymer", "https://jhymer.dev")

Links encontrados: {'links': [{'type': 'Página de la empresa', 'url': 'https://jhymer.dev'}, {'type': 'Página de Cursos', 'url': 'https://jhymer.dev/docker-compose-entorno-desarrollo/'}, {'type': 'Página de Carreras', 'url': 'https://jhymer.dev/herramientas-trabajar-desde-casa/'}]}


# Folleto de Jhymer

## Acerca de Jhymer
**Desarrollo y Más**, fundado por Jhymer Martínez, es una empresa dedicada a la optimización de entornos de desarrollo y la educación en tecnologías modernas. Nuestros artículos abarcan una variedad de temas, desde herramientas de productividad hasta el uso de tecnologías avanzadas como Docker y ReactJS.

## Cultura de la Empresa
En Jhymer, fomentamos un ambiente de colaboración y aprendizaje continuo. Nos enfocamos en el trabajo remoto, apoyando la utilización de herramientas que faciliten la comunicación y la gestión de proyectos. Promovemos un espacio donde cada miembro tiene la libertad de innovar y contribuir a proyectos significativos.

## Servicios y Clientes
Nuestros servicios están orientados a desarrolladores, startups y empresas que buscan optimizar sus procesos de desarrollo. Gracias a nuestros cursos y tutoriales, hemos ayudado a numerosos profesionales a mejorar su competitividad en el mercado laboral.

### Cursos y Packs
En nuestra sección de cursos, ofrecemos capacitaciones como:
- **Optimizar un Entorno de Desarrollo con Docker y Docker-Compose**: Un curso que enseña a los desarrolladores a gestionar sus proyectos de manera eficiente utilizando Docker.
- **JavaScript y ReactJS**: Incluyendo técnicas de pruebas en React, ayudando a los desarrolladores a crear aplicaciones robustas.

## Oportunidades de Carrera
Jhymer busca constantemente talento apasionado por la tecnología. Valoramos la creatividad, la proactividad y la disposición para aprender y adaptarse. Fomentamos el desarrollo profesional a través de la capacitación y la experiencia práctica en proyectos innovadores.

## Herramientas para el Trabajo Remoto
Nuestro equipo utiliza herramientas digitales que facilitan la colaboración y la gestión de proyectos:

- **Slack**: Para comunicación en tiempo real.
- **Zoom**: Para videoconferencias efectivas.
- **Trello**: Para la organización de tareas basadas en la metodología Kanban.

## Conecta con Nosotros
Para descubrir más sobre Jhymer, nuestros servicios o unirte a nuestro equipo, visita nuestras redes sociales y nuestra página web.

- **Facebook**
- **Twitter**
- **Sitio Web**: [Desarrollo y Más](#)

¡Únete a nosotros y transforma tu forma de trabajar en tecnología!

In [35]:
stream_brochure("HuggingFace", "https://huggingface.co")

Links encontrados: {'links': [{'type': 'Página Sobre nosotros', 'url': 'https://huggingface.co/huggingface'}, {'type': 'Página de Empleo', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Página de Modelos', 'url': 'https://huggingface.co/models'}, {'type': 'Página de Datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'Página de Cursos', 'url': 'https://huggingface.co/learn'}, {'type': 'Página de Precios', 'url': 'https://huggingface.co/pricing'}, {'type': 'Página de Espacios', 'url': 'https://huggingface.co/spaces'}]}


# Folleto de Hugging Face 🤗

¡Bienvenido a Hugging Face, donde la inteligencia artificial y la diversión se encuentran! 🎉

## ¿Quiénes Somos?
En Hugging Face, somos la comunidad AI que está construyendo el futuro. Estamos comprometidos a democratizar el aprendizaje automático, un commit a la vez. Nuestro equipo de 222 miembros se dedica a crear un espacio donde la colaboración y la innovación sean el pan de cada día. ¡Tú también puedes ser parte de esta aventura!

## ¿Qué Hacemos?
- **Modelos**: ¡Más de **400k modelos** para que explores y utilices! Desde generación de texto hasta reconocimiento de imágenes, aquí tenemos algo para todos.
- **Datasets**: Con más de **100k datasets**, tu próxima aventura de ML está a solo un clic.
- **Spaces**: Crea, descubre y demuestra tus aplicaciones de ML en nuestros **772 Spaces** activos. ¡Es como un parque de atracciones para los datos!

## Nuestros Clientes 🏢
Más de **50,000 organizaciones** están usando Hugging Face, desde gigantes como Google y Microsoft, hasta startups de fast food digital. No te sorprendas si te encuentras colaborando con algunos de los más grandes nombres en tecnología.

## Cultura de la Empresa 🎈
- Aquí en Hugging Face, influimos más que solo máquinas. Creemos en un ambiente de trabajo divertido, inclusivo y lleno de aprendizaje. ¿Estás listo para unirte a un equipo que ama la inteligencia artificial más de lo que ama su café? ☕
- ¡Nos encanta celebrar cada pequeño logro! Desde lanzamientos de modelos hasta juegos de trivia divertidos durante las reuniones, siempre hay algo por lo que sonreír.

## Oportunidades de Aprendizaje 📚
¿Buscas mejorar tus habilidades en inteligencia artificial? ¡Mira nuestros cursos!
- **Curso de NLP**: Aprende a utilizar todo lo que necesitas sobre Procesamiento de Lenguaje Natural.
- **Curso de RL Profundo**: Ideal para quienes quieren sumergirse en el aprendizaje por refuerzo.
- **Curso de Visión por Computadora**: Aprende a aplicar modelos de ML a tus proyectos visuales.
- **Y muchos más**: Como el curso de audio, el de 3D y hasta cómo integrar AI en tus videojuegos.

## ¿Listo para unirte?
Si sientes que tienes lo que se necesita para construir el futuro con nosotros, ¡revisa nuestras [oportunidades de empleo](https://huggingface.co/jobs)! Nos encantaría contar contigo.

## Precios 💰
- **Siempre gratuito** para explorar, colaborar y aprender.
- **Plan Pro**: Obtén funciones avanzadas por solo **$9/mes**.
- **Plan Enterprise**: Comienza a partir de **$20/usuario/mes** y obtén acceso a todas nuestras funciones personalizadas y soporte prioritario.

## ¡Conéctate Con Nosotros!
- Síguenos en [Twitter](https://twitter.com/huggingface) y [Discord](https://discord.gg/huggingface) para ser parte de nuestra comunidad. ¡No olvides traer tus mejores memes de IA!

### Final Twins 🐢
Hugging Face no es solo una plataforma de ML, ¡es una compañía que se preocupa por las personas y su futuro! Únete a nosotros y ayuda a dar forma al mañana. ¿Te estamos esperando?

---

¡Demos el siguiente paso hacia el futuro juntos! 🚀

In [36]:
stream_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

Links encontrados: {'links': [{'type': 'Página Acerca de', 'url': 'https://cursos.frogamesformacion.com/pages/instructores'}, {'type': 'Página de Certificaciones', 'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'}, {'type': 'Página de Rutas', 'url': 'https://cursos.frogamesformacion.com/pages/rutas'}, {'type': 'Página de Nuestros Clientes', 'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'}, {'type': 'Página de Premios', 'url': 'https://cursos.frogamesformacion.com/pages/premios'}, {'type': 'Cursos de Matemáticas desde Cero', 'url': 'https://cursos.frogamesformacion.com/courses/matematicas-ml-2'}, {'type': 'Cursos de Programación', 'url': 'https://cursos.frogamesformacion.com/collections'}, {'type': 'Página de Afiliados', 'url': 'https://cursos.frogamesformacion.com/pages/afiliados'}]}


# ¡Bienvenido a Frogames Formación! 🐸

### La academia donde aprender es tan divertido como jugar a tus videojuegos favoritos. 🎮

---

## ¿Quiénes somos?

En **Frogames Formación**, somos como los superhéroes de la educación online, pero con menos capas y más código. Fundada por **Juan Gabriel Gomila** y **María Santos**, hemos enseñado a más de **500,000 estudiantes** en toda la comunidad hispanohablante. ¡Y eso es algo que nos hace croar de felicidad! 🐸

---

## ¿Qué hacemos?

Ofrecemos cursos de **Programación de Videojuegos**, **Inteligencia Artificial**, **Machine Learning**, **Desarrollo de Apps**, **Data Science** y mucho más. Te prometemos que aprenderás cosas tan geniales que probablemente querrás gritarlo en las redes sociales. **¡Pero no te olvides de incluir tu certificado de blockchain!** 🏆

---

## Nuestros Cursos Destacados

- **Curso Completo de Unreal Engine 5**: Construye mundos como un mago digital.
- **Machine Learning de la A a la Z**: Conviértete en un gurú de la inteligencia artificial.
- **Curso completo de Python de la A a la Z**: Porque nada dice "tecnología" como una serpiente que programa. 🐍

---

## La Cultura de Frogames

Aquí en Frogames, creemos que **aprender no es aburrido**. ¡Con nuestra comunidad de estudiantes te sentirás entre amigos! Las risas y la colaboración son parte de nuestro ADN. Puedes hacer preguntas, compartir memes (¡sí, también eso cuenta!) y celebrar tus logros.

Si no lo crees, escuchemos a **Javiera Vallejos**, quien dice: "Me encanta ser parte de una comunidad donde se aprende, crece y también se disfruta". ¡Y eso es exactamente lo que buscamos!

---

## Oportunidades por doquier

### ¡Aprovecha nuestro **BLACK FRIDAY**! 🎉
Obtén un **60% de descuento en todos nuestros cursos** usando el **código BLACKFRIDAY**. No dejarás pasar esta oportunidad, ¿verdad?

### Buscando trabajo?
Si alguna vez soñaste con ser un **afiliado de Frogames**, ¡tienes nuestra bendición! Podrás ganar dinero simplemente compartiendo nuestro amor por el aprendizaje. 

---

## Rutas de Aprendizaje 🔍

> **¡Aprende a tu manera!**

- **Matemáticas desde Cero**: ¡Perfecto para aquellos que deben recordar lo que es un número más allá del 1 y el 2!
- **Desarrollo de Videojuegos**: ¿Alguna vez soñaste con crear tu propio juego? ¡Déjanos enseñarte cómo!

---

## ¿Estás listo para saltar? 🐸

No te quedes atrás y únete a la revolución educativa en Frogames Formación. Si no estás seguro de si este es el lugar perfecto para ti... ¡Prueba nuestro curso gratis de trading algorítmico y empieza a ver la magia! ✨

---

**¡Te esperamos en Frogames Formación, donde aprender es una aventura!** 

[¡Inscríbete hoy mismo!](#) 🚀

## Aplicaciones empresariales

En este ejercicio, ampliamos el código del día 1 para realizar múltiples llamadas a LLM y generar un documento.

En términos de técnicas, este es quizás el primer ejemplo de patrones de diseño de Agentic AI, ya que combinamos múltiples llamadas a LLM. Esto se abordará más en la semana 2 y luego volveremos a Agentic AI de manera importante en la semana 8, cuando construyamos una solución Agent completamente autónoma.

En términos de aplicaciones empresariales, generar contenido de esta manera es uno de los casos de uso más comunes. Al igual que con el resumen, esto se puede aplicar a cualquier vertical empresarial. Escriba contenido de marketing, genere un tutorial de producto a partir de una especificación, cree contenido de correo electrónico personalizado y mucho más. Explore cómo puede aplicar la generación de contenido a su negocio e intente crear un prototipo de prueba de concepto.